In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import scale

# Функция для сохранения ответов без переноса строки на конце
def save_answer(filename, answer_str):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(str(answer_str).strip())
    print(f"Файл '{filename}' сохранен с ответом: {answer_str}")

# 1. Загрузка выборки из wine.data
df = pd.read_csv('wine.data', header=None)

# 2. Извлечение классов и признаков
# Класс записан в первом столбце (индекс 0), признаки — во всех остальных
y = df[0]
X = df.drop(columns=[0])

# 3. Настройка кросс-валидации (5 блоков, перемешивание, random_state=42)
# В современных версиях sklearn параметр называется n_splits вместо n_folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# Шаг 4: Поиск оптимального k ДО масштабирования признаков
scores_before = []

for k in range(1, 51):
    knn = KNeighborsClassifier(n_neighbors=k)
    # cross_val_score возвращает массив оценок для каждого из 5 блоков, берем среднее
    cv_score = cross_val_score(knn, X, y, cv=kf, scoring='accuracy').mean()
    scores_before.append((cv_score, k))

# Находим максимум. Если несколько k дают одинаковый результат, 
# max() выберет кортеж с наибольшим k, или мы можем отсортировать аккуратно.
# Для Coursera стандартно берется первое встреченное k с максимальной точностью:
max_score_before, best_k_before = max(scores_before, key=lambda x: (x[0], -x[1]))

save_answer('wine_ans1.txt', best_k_before)
save_answer('wine_ans2.txt', f"{max_score_before:.2f}")

# Шаг 5-6: Поиск оптимального k ПОСЛЕ масштабирования признаков
# Приводим признаки к нулевому среднему и единичному стандартному отклонению
X_scaled = scale(X)

scores_after = []

for k in range(1, 51):
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_score = cross_val_score(knn, X_scaled, y, cv=kf, scoring='accuracy').mean()
    scores_after.append((cv_score, k))

# Находим максимум после масштабирования
max_score_after, best_k_after = max(scores_after, key=lambda x: (x[0], -x[1]))

save_answer('wine_ans3.txt', best_k_after)
save_answer('wine_ans4.txt', f"{max_score_after:.2f}")

# Вывод результатов в консоль для наглядности
print("\n--- Результаты ---")
print(f"До масштабирования: лучшее k = {best_k_before}, точность = {max_score_before:.4f}")
print(f"После масштабирования: лучшее k = {best_k_after}, точность = {max_score_after:.4f}")

Файл 'wine_ans1.txt' сохранен с ответом: 1
Файл 'wine_ans2.txt' сохранен с ответом: 0.73
Файл 'wine_ans3.txt' сохранен с ответом: 29
Файл 'wine_ans4.txt' сохранен с ответом: 0.98

--- Результаты ---
До масштабирования: лучшее k = 1, точность = 0.7305
После масштабирования: лучшее k = 29, точность = 0.9776


In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import scale


def fetch_boston_housing_dataset():
    git_url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
    dataframe = pd.read_csv(git_url)
    
    # Целевой признак в этом датасете называется 'medv' (Median value of owner-occupied homes)
    target = dataframe['medv'].to_numpy()
    features = dataframe.drop(columns=['medv']).to_numpy()
    
    # Обязательное масштабирование признаков по инструкции к лабе
    features_normalized = scale(features)
    
    return features_normalized, target



print("[1/3] Загрузка и нормализация признаков...")
X, y = fetch_boston_housing_dataset()

# Сетка из 200 параметров от 1 до 10 включительно
p_parameters_grid = np.linspace(1.0, 10.0, num=200)

# Кросс-валидация: 5 фолдов, фиксация сида и перемешивание
splitter = KFold(n_splits=5, shuffle=True, random_state=42)

print("[2/3] Сканирование пространства параметров p (200 шагов)...")
grid_scores = []

for current_p in p_parameters_grid:
    # Настройка регрессора строго по ТЗ
    knn_regressor = KNeighborsRegressor(
        n_neighbors=5, 
        weights='distance', 
        metric='minkowski', 
        p=current_p
    )
    
    # Считаем neg_MSE на кросс-валидации
    cv_scores = cross_val_score(
        estimator=knn_regressor, 
        X=X, 
        y=y, 
        cv=splitter, 
        scoring='neg_mean_squared_error'
    )
    
    # Аккумулируем среднее значение
    grid_scores.append(cv_scores.mean())
    
# Превращаем в массив для удобного поиска индекса
grid_scores = np.array(grid_scores)

# Ищем индекс максимального значения (так как ошибка отрицательная, максимум — это лучшая точка)
best_index = np.argmax(grid_scores)
optimal_p = p_parameters_grid[best_index]

# Форматируем ответ (один знак после запятой)
answer_string = f"{optimal_p:.1f}"

print("\n[Результаты сканирования]")
print(f"-> Лучший скор (neg_MSE): {grid_scores[best_index]:.4f}")
print(f"-> Оптимальный параметр p: {answer_string}")

# 3. Запись файла ответа строго без перевода строки на конце
output_path = 'metric_tuning_answer.txt'
with open(output_path, mode='w', encoding='utf-8') as file_stream:
    file_stream.write(answer_string.strip())
    
print(f"\n[3/3] Файл '{output_path}' успешно сгенерирован.")




[1/3] Загрузка и нормализация признаков...
[2/3] Сканирование пространства параметров p (200 шагов)...
